In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd


options = Options()
options.add_argument("--headless")  
options.add_argument("--disable-blink-features=AutomationControlled")
driver = webdriver.Chrome( options=options)


url = "https://www.amazon.com"
driver.get(url)


search_term = "laptop"
search_box = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.ID, "twotabsearchtextbox"))
)
search_box.send_keys(search_term)
search_box.send_keys(Keys.RETURN)

time.sleep(3)  


products = []
try:
    product_elements = WebDriverWait(driver, 20).until(
        EC.presence_of_all_elements_located((By.XPATH, "//div[contains(@class, 's-main-slot')]//div[@data-component-type='s-search-result']"))
    )

    for product in product_elements[:20]:  
        try:
            title = product.find_element(By.XPATH, ".//h2/a/span").text
            price_whole = product.find_element(By.XPATH, ".//span[@class='a-price-whole']").text
            price_fraction = product.find_element(By.XPATH, ".//span[@class='a-price-fraction']").text
            price = f"{price_whole}.{price_fraction}"
            link = product.find_element(By.XPATH, ".//h2/a").get_attribute("href")
            products.append({"Title": title, "Price": price, "Link": link})
        except Exception as e:
            title, price, link = "N/A", "N/A", "N/A"
        
    df = pd.DataFrame(products)
    

finally:
    driver.quit()




In [11]:
df.head()

,Title,Price,Link
0,"HP Stream 14"" HD BrightView Laptop, Intel Cele...",309.00,https://www.amazon.com/HP-Stream-BrightView-N4...
1,"HP 14"" Ultral Light Laptop for Students and Bu...",265.99,https://www.amazon.com/HP-Students-Business-Qu...
2,"HP 14 Laptop, Intel Celeron N4020, 4 GB RAM, 6...",176.95,https://www.amazon.com/HP-Micro-edge-Microsoft...
3,acer Gateway Chromebook 311 CBO311-1H-C1MX Lap...,149.00,https://www.amazon.com/acer-Gateway-Chromebook...
4,"Dell Inspiron Touchscreen Laptop, 15.6"" Busine...",498.66,https://www.amazon.com/Dell-Inspiron-Touchscre...


In [12]:
df.to_csv("amazon-web-scraping.csv", index=False)